# Fitting 02 — Build the Feature Configurations (A, B, C)

Second step of the Fitting stage. Applies the parameters fitted in Fitting 01 to all
three splits and assembles the three nested feature configurations compared in the
thesis: **A** = 10 raw-scale predictors (recoded + imputed only), **B** = the same
information after winsorisation, transformation and two ordinal encodings (no new
columns), **C** = B plus the 12 synthetic features selected in EDA 04. Writes nine
parquet files (`{split}_config{A,B,C}`) and verifies them.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

In [2]:
# Paths, fitted parameters from Fitting 01, and the column lists that define each
# configuration. Configuration B substitutes representations (it does not add columns):
# utilisation and age become ordinal codes, five continuous features become their
# winsorised + transformed versions, the past-due counters stay on their recoded scale.
SEED = 42
ARTIFACTS = Path("../artifacts")
params = json.loads((ARTIFACTS / "pipeline_params.json").read_text())
PAST_DUE_COLS = ["past_due_30_59", "past_due_60_89", "past_due_90"]
CONFIG_A_COLS = ["revolving_utilisation", "age", "debt_ratio", "monthly_income",
                 "open_credit_lines_and_loans", "real_estate_loans_or_lines",
                 "dependents", "past_due_30_59", "past_due_60_89", "past_due_90"]
CONFIG_B_COLS = ["revolving_util_band", "age_lifestage",
                 "debt_ratio_t", "monthly_income_t", "open_credit_lines_and_loans_t",
                 "real_estate_loans_or_lines_t", "dependents_t",
                 "past_due_30_59", "past_due_60_89", "past_due_90"]
CONFIG_C_SYNTHETIC = ["weighted_delinq_score", "combined_delinq_flag", "delinquency_rate",
                      "util_delinq_interaction", "zero_flag_past_due_90",
                      "zero_flag_past_due_60_89", "zero_flag_past_due_30_59",
                      "mahalanobis_log", "multi_extremity_index", "sentinel_flag",
                      "debt_burden_per_dependent", "zero_flag_open_credit_lines"]

In [3]:
# The five preprocessing steps as explicit functions, applied in this fixed order:
# sentinel recode -> impute -> winsorise -> transform -> encode. Each function takes the
# dataframe and the fitted/loaded parameters; nothing is fitted or hardcoded here - the
# transform map and the encoding edges come from pipeline_params.json (written by
# Fitting 01), so this notebook only APPLIES the recorded pipeline. The transforms
# implement the EDA 03 decisions: square root for monthly income (log1p over-corrects
# it), log1p for the other heavy-tailed features, identity for age, real-estate count and
# the past-due counters.
def recode_sentinels(df, sentinel_recode):
    out = df.copy()
    for col, val in sentinel_recode.items():
        out[col] = out[col].replace({96: val, 98: val})
    return out

def impute(df, imputation):
    out = df.copy()
    edges = np.array(imputation["age_decile_edges"])
    # Income medians are indexed by integer age-decile code (matching the labels=False cut
    # fitted in Fitting 01); map by code via a dict so the lookup cannot misalign even if a
    # bin were empty.
    median_by_code = dict(enumerate(imputation["income_medians_per_decile"]))
    decile = pd.cut(out["age"].clip(edges[0], edges[-1]), bins=edges,
                    include_lowest=True, labels=False).astype(int)
    assert decile.between(0, len(median_by_code) - 1).all(), \
        "age decile code out of range when imputing income"
    income_fill = decile.map(median_by_code)
    out["monthly_income"] = out["monthly_income"].fillna(income_fill)
    out["dependents"] = out["dependents"].fillna(imputation["dependents_median"])
    return out

def winsorise(df, winsorisation):
    out = df.copy()
    for col, bounds in winsorisation.items():
        out[col] = out[col].clip(bounds["p01"], bounds["p99"])
    return out

TRANSFORM_FUNCS = {
    "sqrt": lambda s: np.sqrt(s.clip(lower=0)),
    "log1p": lambda s: np.log1p(s.clip(lower=0)),
    "identity": lambda s: s,
}

def transform(df, transform_map):
    out = df.copy()
    for col, name in transform_map.items():
        if col in out.columns and name != "identity":
            out[col] = TRANSFORM_FUNCS[name](out[col])
    return out

def encode_ordinal(series, edges):
    bins = [-np.inf, *edges, np.inf]
    labels = list(range(len(bins) - 1))
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True).astype(int)

In [4]:
# Apply the pipeline to one split and return a "wide" dataframe holding everything the
# three configurations draw from: the recoded + imputed raw columns (configuration A),
# the winsorised + transformed columns with a `_t` suffix (configuration B), the two
# ordinal encodings, and the sentinel flag. The encodings are computed from the RAW
# utilisation and age values on purpose - winsorising first would collapse the
# over-100% utilisation band, which carries a 6x default-rate lift (EDA 03). All
# transform/encoding choices come from the loaded params (no hardcoded literals).
def build_wide(raw_df, params):
    enc = params["encoding"]
    sentinel_flag = raw_df[PAST_DUE_COLS].isin([96, 98]).any(axis=1).astype(int)
    df = recode_sentinels(raw_df, params["sentinel_recode"])
    util_band = encode_ordinal(df["revolving_utilisation"], enc["revolving_util_band_edges"])
    age_stage = encode_ordinal(df["age"], enc["age_lifestage_edges"])
    df = impute(df, params["imputation"])
    transformed = transform(winsorise(df, params["winsorisation"]), params["transform_map"])
    wide = pd.DataFrame(index=raw_df.index)
    wide["target"] = raw_df["target"]
    for col in CONFIG_A_COLS:
        wide[col] = df[col]
    for col in ["debt_ratio", "monthly_income", "open_credit_lines_and_loans",
                "real_estate_loans_or_lines", "dependents"]:
        wide[col + "_t"] = transformed[col]
    wide["revolving_util_band"] = util_band
    wide["age_lifestage"] = age_stage
    wide["sentinel_flag"] = sentinel_flag
    return wide

wide = {name: build_wide(pd.read_parquet(ARTIFACTS / f"{name}.parquet"), params)
        for name in ("train", "val", "test")}
print({name: w.shape for name, w in wide.items()})

{'train': (89634, 19), 'val': (29878, 19), 'test': (29879, 19)}


In [5]:
# Fit the parameters that two synthetic features need - on the TRAINING split only - and
# append them to pipeline_params.json so the artifact is a complete, replayable record.
# The Mahalanobis distance needs the training mean and inverse covariance of the ten raw
# predictors; the multi-extremity index needs each predictor's training 99th percentile.
# All other synthetic features are computed row by row and need no fitting.
train_raw = wide["train"][CONFIG_A_COLS]
mahalanobis_mean = train_raw.to_numpy(dtype=float).mean(axis=0)
covariance = np.cov(train_raw.to_numpy(dtype=float), rowvar=False) + 1e-6 * np.eye(len(CONFIG_A_COLS))
mahalanobis_cov_inv = np.linalg.pinv(covariance)
extremity_p99 = {col: float(train_raw[col].quantile(0.99)) for col in CONFIG_A_COLS}

params["synthetic_params"] = {
    "mahalanobis_mean": mahalanobis_mean.tolist(),
    "mahalanobis_cov_inv": mahalanobis_cov_inv.tolist(),
    "extremity_p99": extremity_p99,
}
(ARTIFACTS / "pipeline_params.json").write_text(json.dumps(params, indent=2))
print("synthetic-feature parameters fitted on the training split and appended to "
      "pipeline_params.json")

synthetic-feature parameters fitted on the training split and appended to pipeline_params.json


In [6]:
# Build the 12 configuration-C synthetic features selected in EDA 04. The ratio features
# use winsorised (but untransformed) inputs, because a ratio of log-transformed values
# has no economic meaning. The Mahalanobis distance and the extremity count use the
# train-fitted parameters loaded from pipeline_params.json (passed in via `params`, not
# read from notebook globals), so this function is reproducible from the artifact alone.
def build_synthetics(w, params):
    pd30, pd60, pd90 = w["past_due_30_59"], w["past_due_60_89"], w["past_due_90"]
    bounds = params["winsorisation"]
    sp = params["synthetic_params"]
    mahalanobis_mean = np.asarray(sp["mahalanobis_mean"], dtype=float)
    mahalanobis_cov_inv = np.asarray(sp["mahalanobis_cov_inv"], dtype=float)
    extremity_p99 = sp["extremity_p99"]
    s = pd.DataFrame(index=w.index)
    s["weighted_delinq_score"] = 1 * pd30 + 2 * pd60 + 3 * pd90
    s["combined_delinq_flag"] = ((pd30 > 0).astype(int) + (pd60 > 0).astype(int)
                                 + (pd90 > 0).astype(int))
    s["zero_flag_past_due_30_59"] = (pd30 == 0).astype(int)
    s["zero_flag_past_due_60_89"] = (pd60 == 0).astype(int)
    s["zero_flag_past_due_90"] = (pd90 == 0).astype(int)
    s["util_delinq_interaction"] = w["revolving_util_band"] * s["weighted_delinq_score"]
    debt_w = w["debt_ratio"].clip(bounds["debt_ratio"]["p01"], bounds["debt_ratio"]["p99"])
    dep_w = w["dependents"].clip(bounds["dependents"]["p01"], bounds["dependents"]["p99"])
    lines_w = w["open_credit_lines_and_loans"].clip(
        bounds["open_credit_lines_and_loans"]["p01"],
        bounds["open_credit_lines_and_loans"]["p99"])
    s["debt_burden_per_dependent"] = debt_w / dep_w.clip(lower=1)
    s["delinquency_rate"] = s["weighted_delinq_score"] / lines_w.clip(lower=1)
    s["sentinel_flag"] = w["sentinel_flag"]
    s["zero_flag_open_credit_lines"] = (w["open_credit_lines_and_loans"] == 0).astype(int)
    s["multi_extremity_index"] = sum((w[c] > extremity_p99[c]).astype(int)
                                     for c in CONFIG_A_COLS)
    centred = w[CONFIG_A_COLS].to_numpy(dtype=float) - mahalanobis_mean
    d2 = np.einsum("ij,jk,ik->i", centred, mahalanobis_cov_inv, centred)
    s["mahalanobis_log"] = np.log1p(np.clip(d2, 0, None))
    return s[CONFIG_C_SYNTHETIC]

In [7]:
# Assemble and save the nine feature matrices: each split in configuration A (10 raw
# predictors), B (10 substituted representations) and C (B + 12 synthetics). These
# parquet files are the only data interface between the Fitting notebooks and the models.
for name, w in wide.items():
    synthetics = build_synthetics(w, params)
    config_a = pd.concat([w["target"], w[CONFIG_A_COLS]], axis=1)
    config_b = pd.concat([w["target"], w[CONFIG_B_COLS]], axis=1)
    config_c = pd.concat([config_b, synthetics], axis=1)
    config_a.to_parquet(ARTIFACTS / f"{name}_configA.parquet")
    config_b.to_parquet(ARTIFACTS / f"{name}_configB.parquet")
    config_c.to_parquet(ARTIFACTS / f"{name}_configC.parquet")
    print(f"{name}: A {config_a.shape}  B {config_b.shape}  C {config_c.shape}")

train: A (89634, 11)  B (89634, 11)  C (89634, 23)


val: A (29878, 11)  B (29878, 11)  C (29878, 23)
test: A (29879, 11)  B (29879, 11)  C (29879, 23)


In [8]:
# Verification - the leakage-safety and integrity claims proven in code, not just asserted
# in prose. Four checks:
#  (1) INTEGRITY: every matrix is fully numeric, complete, with the intended column counts.
#  (2) SPLIT DISJOINTNESS: the raw train/val/test rows are pairwise disjoint and together
#      partition the deduplicated data, so no record can leak across the split boundary.
#  (3) IDEMPOTENT REBUILD: re-applying the pipeline from the loaded params reproduces the
#      saved config matrices exactly (the pipeline is a pure function of params + raw data).
#  (4) NO-LEAK NESTING: B's columns are a subset of C's, and none of the 12 synthetic
#      columns appear in A or B (the enrichment only ever adds, never contaminates).

# (1) integrity
for name in ("train", "val", "test"):
    for cfg, n_cols in (("A", 11), ("B", 11), ("C", 23)):
        m = pd.read_parquet(ARTIFACTS / f"{name}_config{cfg}.parquet")
        assert m.shape[1] == n_cols, f"{name}_{cfg}: wrong column count {m.shape[1]}"
        assert m.isna().sum().sum() == 0, f"{name}_{cfg}: missing values present"
        assert all(np.issubdtype(t, np.number) for t in m.dtypes), f"{name}_{cfg}: non-numeric"

# (2) split disjointness + partition (row hashes; rows are unique after dedup in Fitting 01)
raw = {n: pd.read_parquet(ARTIFACTS / f"{n}.parquet") for n in ("train", "val", "test")}
row_hashes = {n: set(pd.util.hash_pandas_object(r, index=False)) for n, r in raw.items()}
assert row_hashes["train"].isdisjoint(row_hashes["val"]), "train/val overlap"
assert row_hashes["train"].isdisjoint(row_hashes["test"]), "train/test overlap"
assert row_hashes["val"].isdisjoint(row_hashes["test"]), "val/test overlap"
total = sum(len(r) for r in raw.values())
assert len(set().union(*row_hashes.values())) == total, "splits do not partition the data"

# (3) idempotent rebuild from the loaded params
for name in ("train", "val", "test"):
    w2 = build_wide(pd.read_parquet(ARTIFACTS / f"{name}.parquet"), params)
    rebuilt = pd.concat([w2["target"], w2[CONFIG_B_COLS], build_synthetics(w2, params)], axis=1)
    saved = pd.read_parquet(ARTIFACTS / f"{name}_configC.parquet")
    pd.testing.assert_frame_equal(rebuilt.reset_index(drop=True), saved.reset_index(drop=True))

# (4) no-leak nesting
cfgA_cols = set(pd.read_parquet(ARTIFACTS / "train_configA.parquet").columns)
cfgB_cols = set(pd.read_parquet(ARTIFACTS / "train_configB.parquet").columns)
cfgC_cols = set(pd.read_parquet(ARTIFACTS / "train_configC.parquet").columns)
assert cfgB_cols <= cfgC_cols, "config B is not nested in config C"
assert not (set(CONFIG_C_SYNTHETIC) & cfgA_cols), "synthetic column leaked into config A"
assert not (set(CONFIG_C_SYNTHETIC) & cfgB_cols), "synthetic column leaked into config B"

print("all nine matrices verified: integrity, split disjointness, idempotent rebuild, "
      "no-leak nesting")

all nine matrices verified: integrity, split disjointness, idempotent rebuild, no-leak nesting


In [9]:
# Smoke test: a plain Logistic Regression trained on each configuration and scored on the
# validation split. Each config must clear a basic discrimination floor (AUC > 0.70), and
# the AUC should rise A -> B -> C if the preprocessing (B) and the synthetic features (C)
# add real signal. The A->B->C values are recorded (not just printed); a non-monotone
# step is surfaced as a warning rather than a crash, because small reversals are within
# noise. This is only a sanity check - the proper model comparison happens in Fitting 03.
rows = []
for cfg in ("A", "B", "C"):
    train_m = pd.read_parquet(ARTIFACTS / f"train_config{cfg}.parquet")
    val_m = pd.read_parquet(ARTIFACTS / f"val_config{cfg}.parquet")
    model = Pipeline([("scaler", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                 random_state=SEED))])
    model.fit(train_m.drop(columns="target"), train_m["target"])
    proba = model.predict_proba(val_m.drop(columns="target"))[:, 1]
    auc = roc_auc_score(val_m["target"], proba)
    assert auc > 0.70, f"config {cfg}: smoke-test AUC {auc:.4f} below the 0.70 floor"
    rows.append({"config": cfg, "val_auc": auc})

smoke = pd.DataFrame(rows)
aucs = smoke.set_index("config")["val_auc"]
if not (aucs["A"] <= aucs["B"] <= aucs["C"]):
    print(f"WARNING: smoke-test AUC not monotone A<=B<=C: {aucs.round(4).to_dict()}")
else:
    print(f"smoke-test AUC monotone A<=B<=C: {aucs.round(4).to_dict()}")
smoke.round(4)

smoke-test AUC monotone A<=B<=C: {'A': 0.8201, 'B': 0.8496, 'C': 0.8577}


,config,val_auc
0,A,0.8201
1,B,0.8496
2,C,0.8577
